# Loan Approval Prediction Training

Notebook ini berisi proses training model Random Forest untuk dashboard prediksi persetujuan pinjaman.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

BASE_DIR = Path('..')
DATASET_PATH = BASE_DIR / 'loan_approval_dataset.csv'
MODEL_PATH = BASE_DIR / 'random_forest_loan_model.pkl'
EVALUATION_PATH = BASE_DIR / 'model_evaluation_results.csv'
FEATURE_IMPORTANCE_PATH = BASE_DIR / 'feature_importance_results.csv'


## 1. Load dan Cleaning Dataset

In [ ]:
df = pd.read_csv(DATASET_PATH)

# Membersihkan spasi pada nama kolom dan isi data kategori
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

df.head()

In [ ]:
print('Jumlah data:', df.shape)
print('\nMissing value:')
print(df.isnull().sum())
print('\nDistribusi target:')
print(df['loan_status'].value_counts())

## 2. Menyiapkan Fitur dan Target

In [ ]:
df_model = df.drop(columns=['loan_id'])
df_model['loan_status'] = df_model['loan_status'].map({'Approved': 1, 'Rejected': 0})

X = df_model.drop(columns=['loan_status'])
y = df_model['loan_status']

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print('Fitur numerik:', numeric_features)
print('Fitur kategorikal:', categorical_features)

## 3. Preprocessing dan Split Data

In [ ]:
try:
    onehot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', onehot, categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Jumlah data training:', X_train.shape)
print('Jumlah data testing:', X_test.shape)

## 4. Training Random Forest dan SVM

In [ ]:
rf_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight='balanced'
        ))
    ]
)

svm_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', SVC(
            kernel='rbf',
            probability=True,
            random_state=42,
            class_weight='balanced'
        ))
    ]
)

rf_model.fit(X_train, y_train)
svm_model.fit(X_train, y_train)
print('Training selesai.')

## 5. Evaluasi Model

In [ ]:
models = {
    'Random Forest': rf_model,
    'SVM': svm_model
}

evaluation_results = []

for model_name, model in models.items():
    y_pred = model.predict(X_test)
    evaluation_results.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred)
    })
    print('=' * 50)
    print(model_name)
    print('=' * 50)
    print(classification_report(y_test, y_pred, target_names=['Rejected', 'Approved']))

evaluation_df = pd.DataFrame(evaluation_results)
evaluation_df

## 6. Feature Importance Random Forest

In [ ]:
preprocessor_fitted = rf_model.named_steps['preprocessor']
onehot_encoder = preprocessor_fitted.named_transformers_['cat']

try:
    categorical_names = onehot_encoder.get_feature_names_out(categorical_features)
except AttributeError:
    categorical_names = onehot_encoder.get_feature_names(categorical_features)

all_feature_names = np.concatenate([numeric_features, categorical_names])
importances = rf_model.named_steps['classifier'].feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

feature_importance_df.head(10)

## 7. Simpan Model dan Hasil Evaluasi

In [ ]:
joblib.dump(rf_model, MODEL_PATH)
evaluation_df.to_csv(EVALUATION_PATH, index=False)
feature_importance_df.to_csv(FEATURE_IMPORTANCE_PATH, index=False)

print('Model dan hasil evaluasi berhasil disimpan.')

## 8. Contoh Prediksi

In [ ]:
new_data = pd.DataFrame({
    'no_of_dependents': [1],
    'education': ['Graduate'],
    'self_employed': ['No'],
    'income_annum': [8000000],
    'loan_amount': [20000000],
    'loan_term': [8],
    'cibil_score': [780],
    'residential_assets_value': [9000000],
    'commercial_assets_value': [3000000],
    'luxury_assets_value': [12000000],
    'bank_asset_value': [5000000]
})

prediction = rf_model.predict(new_data)[0]
proba = rf_model.predict_proba(new_data)[0]

print('Hasil Prediksi:', 'Approved' if prediction == 1 else 'Rejected')
print('Probabilitas Rejected:', proba[0])
print('Probabilitas Approved:', proba[1])